In [43]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
from collections import defaultdict

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import label_binarize

In [45]:
DATA_DIR = Path("data/raw_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATA_DIR = Path("data/final_data")
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
ELO_PATH = DATA_DIR / "elo_ratings" / "eloratings.csv"

In [46]:
df = pd.read_csv(f"{DATA_DIR}/international_matches/results.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  object 
 1   home_team   49477 non-null  object 
 2   away_team   49477 non-null  object 
 3   home_score  49453 non-null  float64
 4   away_score  49453 non-null  float64
 5   tournament  49477 non-null  object 
 6   city        49477 non-null  object 
 7   country     49477 non-null  object 
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


In [48]:
df.isna().sum()

date           0
home_team      0
away_team      0
home_score    24
away_score    24
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [49]:
# What are these NA values ??
missing_score_matches = df[df["home_score"].isna() | df["away_score"].isna()].copy()
missing_score_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49453,2026-06-24,Mexico,Czech Republic,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49454,2026-06-24,South Africa,South Korea,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49455,2026-06-24,Canada,Switzerland,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49456,2026-06-24,Bosnia and Herzegovina,Qatar,NaN,NaN,FIFA World Cup,Seattle,United States,True
49457,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True


In [50]:
# Drop these future values because we don't need them
df = df.dropna(subset=["home_score", "away_score"]).copy()

In [51]:
# Make the actual dataframe from this now
matches = pd.DataFrame()

matches["match_id"] = range(1, len(df) + 1)
matches["date"] = pd.to_datetime(df["date"])
matches["team_a"] = df["home_team"].str.strip()
matches["team_b"] = df["away_team"].str.strip()
matches["team_a_score"] = df["home_score"].astype(int)
matches["team_b_score"] = df["away_score"].astype(int)
matches["tournament"] = df["tournament"].str.strip()
matches["city"] = df["city"].str.strip()
matches["country"] = df["country"].str.strip()
matches["neutral"] = df["neutral"].astype(bool)
matches["is_team_a_home"] = 1

In [52]:
matches.head()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,is_team_a_home
0,1,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1
1,2,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1
2,3,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1
3,4,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1
4,5,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1


In [53]:
targets = []

for a_score, b_score in zip(matches["team_a_score"], matches["team_b_score"]):
    if a_score > b_score:
        targets.append(1)
    elif a_score < b_score:
        targets.append(2)
    else:
        targets.append(0)

matches["target"] = targets
matches[["target", "team_a_score", "team_b_score"]].head()

,target,team_a_score,team_b_score
0,0,0,0
1,1,4,2
2,1,2,1
3,0,2,2
4,1,3,0


In [54]:
matches["tournament_lower"]=matches["tournament"].str.lower()
matches["is_world_cup_qualifier"] = (
    matches["tournament_lower"].str.contains(
        r"fifa world cup.*qualification|fifa world cup.*qualifier",
        regex=True,
        na=False
    )
).astype(int)

matches["is_world_cup"] = (
    matches["tournament_lower"].eq("fifa world cup")
).astype(int)

In [55]:
matches["is_continental"] = (
    matches["tournament_lower"].str.contains(
        r"uefa euro|copa américa|copa america|african cup of nations|afc asian cup|concacaf gold cup|ofc nations cup",
        regex=True,
        na=False
    )
).astype(int)

matches["is_friendly"] = (
    matches["tournament_lower"].str.contains(
        r"friendly",
        regex=True,
        na=False
    )
).astype(int)

In [56]:
matches[
    [
        "is_world_cup",
        "is_world_cup_qualifier",
        "is_continental",
        "is_friendly"
    ]
].sum()

is_world_cup               1012
is_world_cup_qualifier     8771
is_continental             8511
is_friendly               18388
dtype: int64

In [57]:
matches.head()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,is_team_a_home,target,tournament_lower,is_world_cup_qualifier,is_world_cup,is_continental,is_friendly
0,1,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1,0,friendly,0,0,0,1
1,2,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1,1,friendly,0,0,0,1
2,3,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1,1,friendly,0,0,0,1
3,4,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1,0,friendly,0,0,0,1
4,5,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1,1,friendly,0,0,0,1


In [58]:
team_a_history = matches[["team_a","team_b", "team_a_score", "team_b_score", "target", "match_id", "date"]].copy()
team_b_history = matches[["team_a","team_b", "team_a_score", "team_b_score", "target", "match_id", "date"]].copy()

team_a_history = team_a_history.rename(
    columns = {
        "team_a":"team",
        "team_b":"opponent",
        "team_a_score":"goals_for",
        "team_b_score":"goals_against"
    }
)

team_b_history = team_b_history.rename(
    columns = {
        "team_a":"opponent",
        "team_b":"team",
        "team_a_score":"goals_against",
        "team_b_score":"goals_for"
    }
)

In [59]:
def get_points_for_team(match_row, team_name):
    if match_row["team_a"] == team_name:
        goals_for = match_row["team_a_score"]
        goals_against = match_row["team_b_score"]
    else:
        goals_for = match_row["team_b_score"]
        goals_against = match_row["team_a_score"]

    if goals_for > goals_against:
        return 3
    elif goals_for == goals_against:
        return 1
    else:
        return 0

In [60]:
team_a_history = matches[
    ["match_id", "date", "team_a", "team_b", "team_a_score", "team_b_score"]
].copy()

team_a_history = team_a_history.rename(
    columns={
        "team_a": "team",
        "team_b": "opponent",
        "team_a_score": "goals_for",
        "team_b_score": "goals_against",
    }
)

team_b_history = matches[
    ["match_id", "date", "team_a", "team_b", "team_a_score", "team_b_score"]
].copy()

team_b_history = team_b_history.rename(
    columns={
        "team_b": "team",
        "team_a": "opponent",
        "team_b_score": "goals_for",
        "team_a_score": "goals_against",
    }
)

team_history = pd.concat([team_a_history, team_b_history], ignore_index=True)

team_history["points"] = np.where(
    team_history["goals_for"] > team_history["goals_against"], 3,
    np.where(
        team_history["goals_for"] == team_history["goals_against"], 1,
        0
    )
)

team_history = team_history.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

team_history["points_last_5"] = (
    team_history
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

In [61]:
team_a_points = team_history[
    ["match_id", "team", "points_last_5"]
].rename(
    columns={
        "team": "team_a",
        "points_last_5": "team_a_points_last_5"
    }
)

team_b_points = team_history[
    ["match_id", "team", "points_last_5"]
].rename(
    columns={
        "team": "team_b",
        "points_last_5": "team_b_points_last_5"
    }
)

matches = matches.merge(
    team_a_points,
    on=["match_id", "team_a"],
    how="left"
)

matches = matches.merge(
    team_b_points,
    on=["match_id", "team_b"],
    how="left"
)

matches["points_last_5_diff"] = (
    matches["team_a_points_last_5"] - matches["team_b_points_last_5"]
)

In [62]:
matches[
    [
        "date",
        "team_a",
        "team_b",
        "team_a_points_last_5",
        "team_b_points_last_5",
        "points_last_5_diff"
    ]
].head()

,date,team_a,team_b,team_a_points_last_5,team_b_points_last_5,points_last_5_diff
0,1872-11-30,Scotland,England,NaN,NaN,NaN
1,1873-03-08,England,Scotland,1.0,1.0,0.0
2,1874-03-07,Scotland,England,1.0,4.0,-3.0
3,1875-03-06,England,Scotland,4.0,4.0,0.0
4,1876-03-04,Scotland,England,5.0,5.0,0.0


In [63]:
def add_pre_match_elo(matches_df, elo_df, team_col, new_col):
    output_parts = []

    temp = matches_df[["match_id", "date", team_col]].copy()
    temp = temp.rename(columns={team_col: "team"})

    for team_name, team_matches in temp.groupby("team"):
        team_elo = elo_df[elo_df["team"] == team_name][["date", "rating"]].copy()

        if team_elo.empty:
            team_matches[new_col] = pd.NA
        else:
            team_matches = team_matches.sort_values("date")
            team_elo = team_elo.sort_values("date")

            merged = pd.merge_asof(
                team_matches,
                team_elo,
                on="date",
                direction="backward",
                allow_exact_matches=False
            )

            team_matches[new_col] = merged["rating"]

        output_parts.append(team_matches[["match_id", new_col]])

    return pd.concat(output_parts, ignore_index=True)

In [64]:
elo = pd.read_csv(ELO_PATH)

elo["date"] = pd.to_datetime(elo["date"], format="mixed", errors="coerce")
elo["team"] = elo["team"].str.strip()
elo["rating"] = pd.to_numeric(elo["rating"], errors="coerce")

matches["date"] = pd.to_datetime(matches["date"])
matches["team_a"] = matches["team_a"].str.strip()
matches["team_b"] = matches["team_b"].str.strip()

elo = elo.sort_values(["team", "date"]).reset_index(drop=True)
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True)

team_a_elo = add_pre_match_elo(matches, elo, "team_a", "team_a_elo")
team_b_elo = add_pre_match_elo(matches, elo, "team_b", "team_b_elo")

matches = matches.merge(team_a_elo, on="match_id", how="left")
matches = matches.merge(team_b_elo, on="match_id", how="left")

matches["elo_diff"] = matches["team_a_elo"] - matches["team_b_elo"]

C:\Users\SAMARTH\AppData\Local\Temp\ipykernel_23652\3641760760.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(output_parts, ignore_index=True)
C:\Users\SAMARTH\AppData\Local\Temp\ipykernel_23652\3641760760.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(output_parts, ignore_index=True)


In [65]:
# Basic pi ratings
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True).copy()

INITIAL_PI = 0.0
LEARNING_RATE = 0.05
HOME_ADVANTAGE = 0.25
MAX_GOAL_DIFF = 5

team_pi = defaultdict(lambda: INITIAL_PI)

team_a_pi_list = []
team_b_pi_list = []
pi_diff_list = []
expected_gd_list = []
actual_gd_list = []
pi_error_list = []

for _, row in matches.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]

    team_a_pi = team_pi[team_a]
    team_b_pi = team_pi[team_b]

    home_advantage = 0 if row["neutral"] else HOME_ADVANTAGE

    expected_gd = team_a_pi - team_b_pi + home_advantage

    actual_gd = row["team_a_score"] - row["team_b_score"]
    actual_gd = np.clip(actual_gd, -MAX_GOAL_DIFF, MAX_GOAL_DIFF)

    error = actual_gd - expected_gd

    team_a_pi_list.append(team_a_pi)
    team_b_pi_list.append(team_b_pi)
    pi_diff_list.append(team_a_pi - team_b_pi)
    expected_gd_list.append(expected_gd)
    actual_gd_list.append(actual_gd)
    pi_error_list.append(error)

    update = LEARNING_RATE * error

    team_pi[team_a] += update
    team_pi[team_b] -= update

matches["team_a_pi"] = team_a_pi_list
matches["team_b_pi"] = team_b_pi_list
matches["pi_diff"] = pi_diff_list

matches["pi_expected_goal_diff"] = expected_gd_list
matches["pi_actual_goal_diff"] = actual_gd_list
matches["pi_error"] = pi_error_list

In [66]:
matches.tail()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,...,points_last_5_diff,team_a_elo,team_b_elo,elo_diff,team_a_pi,team_b_pi,pi_diff,pi_expected_goal_diff,pi_actual_goal_diff,pi_error
49448,49449,2026-06-22,Jordan,Algeria,1,2,FIFA World Cup,Santa Clara,United States,True,...,-8.0,NaN,NaN,NaN,1.219519,2.087010,-0.867491,-0.867491,-1,-0.132509
49449,49450,2026-06-23,Portugal,Uzbekistan,5,0,FIFA World Cup,Houston,United States,True,...,7.0,NaN,NaN,NaN,2.760579,1.449828,1.310752,1.310752,5,3.689248
49450,49451,2026-06-23,Colombia,DR Congo,1,0,FIFA World Cup,Zapopan,Mexico,True,...,1.0,NaN,NaN,NaN,3.005946,1.288544,1.717402,1.717402,1,-0.717402
49451,49452,2026-06-23,England,Ghana,0,0,FIFA World Cup,Foxborough,United States,True,...,6.0,NaN,NaN,NaN,2.988931,0.984624,2.004306,2.004306,0,-2.004306
49452,49453,2026-06-23,Panama,Croatia,0,1,FIFA World Cup,Toronto,Canada,True,...,1.0,NaN,NaN,NaN,1.282684,2.194090,-0.911405,-0.911405,-1,-0.088595


In [67]:
feature_cols = [
    "neutral",
    "is_team_a_home",
    "is_friendly",
    "is_world_cup",
    "is_world_cup_qualifier",
    "is_continental",

    "team_a_points_last_5",
    "team_b_points_last_5",
    "points_last_5_diff",

    "team_a_elo",
    "team_b_elo",
    "elo_diff",

    "team_a_pi",
    "team_b_pi",
    "pi_diff",
]

In [68]:
model_df = matches[feature_cols + ["target", "date"]].copy()
model_df = model_df.dropna(subset=["target"]).copy()
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)

In [69]:
model_df.to_csv(FINAL_DATA_DIR / "model_df.csv", index=False)

In [70]:
split_index = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]
y_test = test_df["target"]

In [71]:
majority_class = y_train.mode()[0]

y_pred_baseline = np.full(shape=len(y_test), fill_value=majority_class)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print("Majority class:", majority_class)
print("Baseline accuracy:", baseline_accuracy)

print(classification_report(y_test, y_pred_baseline))

Majority class: 1
Baseline accuracy: 0.47659488423819635
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2305
           1       0.48      1.00      0.65      4714
           2       0.00      0.00      0.00      2872

    accuracy                           0.48      9891
   macro avg       0.16      0.33      0.22      9891
weighted avg       0.23      0.48      0.31      9891



c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [72]:
class_order = [0, 1, 2]

train_class_probs = (
    y_train.value_counts(normalize=True)
    .reindex(class_order, fill_value=0)
    .values
)

y_proba_baseline = np.tile(train_class_probs, (len(y_test), 1))

baseline_log_loss = log_loss(
    y_test,
    y_proba_baseline,
    labels=class_order
)

print("Baseline log loss:", baseline_log_loss)

Baseline log loss: 1.0522739971799695


In [73]:
# Logistic Regression Model
log_reg_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            multi_class="multinomial",
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

log_reg_model.fit(X_train, y_train)

c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [74]:
y_pred = log_reg_model.predict(X_test)
y_proba = log_reg_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_proba, labels=log_reg_model.classes_)

print("Accuracy:", accuracy)
print("Log loss:", loss)

print(classification_report(y_test, y_pred))

Accuracy: 0.575877059953493
Log loss: 0.8989147135970486
              precision    recall  f1-score   support

           0       0.31      0.27      0.29      2305
           1       0.70      0.69      0.69      4714
           2       0.57      0.63      0.60      2872

    accuracy                           0.58      9891
   macro avg       0.52      0.53      0.53      9891
weighted avg       0.57      0.58      0.57      9891



In [75]:
def multiclass_brier_score(y_true, y_proba, class_order):
    y_true_onehot = label_binarize(y_true, classes=class_order)
    return np.mean(np.sum((y_proba - y_true_onehot) ** 2, axis=1))


def ranked_probability_score(y_true, y_proba, class_order):
    y_true_onehot = label_binarize(y_true, classes=class_order)

    cumulative_proba = np.cumsum(y_proba, axis=1)
    cumulative_true = np.cumsum(y_true_onehot, axis=1)

    rps = np.mean(
        np.sum((cumulative_proba - cumulative_true) ** 2, axis=1)
        / (len(class_order) - 1)
    )

    return rps

In [76]:
def evaluate_model(model_name, y_test, y_pred, y_proba):
    acc = accuracy_score(y_test, y_pred)
    loss = log_loss(y_test, y_proba, labels=class_order)
    brier = multiclass_brier_score(y_test, y_proba, class_order)
    rps = ranked_probability_score(y_test, y_proba, class_order)

    print(f"\n{model_name}")
    print("-" * len(model_name))
    print("Accuracy:", acc)
    print("Log loss:", loss)
    print("Brier score:", brier)
    print("Ranked probability score:", rps)
    print(classification_report(y_test, y_pred))

In [77]:
# Random Forest
rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

evaluate_model("Random Forest", y_test, rf_pred, rf_proba)


Random Forest
-------------
Accuracy: 0.5727428975836619
Log loss: 0.9122085540175401
Brier score: 0.5393216272927847
Ranked probability score: 0.16875183578043587
              precision    recall  f1-score   support

           0       0.31      0.29      0.30      2305
           1       0.70      0.68      0.69      4714
           2       0.57      0.63      0.60      2872

    accuracy                           0.57      9891
   macro avg       0.53      0.53      0.53      9891
weighted avg       0.57      0.57      0.57      9891



In [78]:
# XGBoost
xgb_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_pred = xgb_model.predict(X_test)

xgb_proba = xgb_model.predict_proba(X_test)

evaluate_model("XGBoost", y_test, xgb_pred, xgb_proba)


XGBoost
-------
Accuracy: 0.6001415428167021
Log loss: 0.8749133357248872
Brier score: 0.5142525850269591
Ranked probability score: 0.16338131399529898
              precision    recall  f1-score   support

           0       0.21      0.00      0.01      2305
           1       0.60      0.89      0.72      4714
           2       0.59      0.60      0.60      2872

    accuracy                           0.60      9891
   macro avg       0.47      0.50      0.44      9891
weighted avg       0.51      0.60      0.52      9891



In [79]:
# CatBoost
cat_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", CatBoostClassifier(
            loss_function="MultiClass",
            iterations=300,
            depth=4,
            learning_rate=0.05,
            random_seed=42,
            verbose=False
        ))
    ]
)

cat_model.fit(X_train, y_train)

cat_pred = cat_model.predict(X_test).flatten()

cat_proba = cat_model.predict_proba(X_test)

evaluate_model("CatBoost", y_test, cat_pred, cat_proba)


CatBoost
--------
Accuracy: 0.6013547669598625
Log loss: 0.8730146808123173
Brier score: 0.5130012430232137
Ranked probability score: 0.1630268537405836
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2305
           1       0.61      0.89      0.72      4714
           2       0.59      0.61      0.60      2872

    accuracy                           0.60      9891
   macro avg       0.40      0.50      0.44      9891
weighted avg       0.46      0.60      0.52      9891

